# Learning to Rank (CatBoost, LETOR, WikIR, MIRAGE)

Потребуются следующие данные:

- Internet Mathematics 2009 dataset (MS LETOR формат после преобразования)
- [WikIR/en1k](https://github.com/getalp/wiIR)
- [MIRAGE](https://github.com/nlpai-lab/MIRAGE)

Все данные предполагаются в директории `../data/`.

Для работы потребуется установить:

```bash
pip install catboost rank_bm25 scikit-learn pandas numpy tqdm requests
```

In [1]:
import json
import os
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from catboost import CatBoostRanker, Pool
from catboost.datasets import msrank_10k
from rank_bm25 import BM25Okapi
from sklearn.metrics import ndcg_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm

## CatBoost на MS LETOR

### Загрузка данных

In [2]:
train_df, test_df = msrank_10k()

X_train = train_df.drop([0, 1], axis=1).values
y_train = train_df[0].values
qid_train = train_df[1].values

X_test = test_df.drop([0, 1], axis=1).values
y_test = test_df[0].values
qid_test = test_df[1].values

train_pool = Pool(X_train, y_train, group_id=qid_train)
test_pool = Pool(X_test, y_test, group_id=qid_test)

### Обучение CatBoost + оценка nDCG

In [3]:
def compute_group_ndcg(y_true, y_pred, qids):
    result = []

    for q in np.unique(qids):
        mask = qids == q

        if np.sum(mask) < 2:
            continue

        if np.sum(y_true[mask]) == 0:
            continue

        result.append(ndcg_score([y_true[mask]], [y_pred[mask]]))

    return np.mean(result) if result else 0.0


random_preds = np.random.rand(len(y_test))
print('Random, nDCG:', compute_group_ndcg(y_test, random_preds, qid_test))

Random, nDCG: 0.6336086798419892


In [4]:
model = CatBoostRanker(
    iterations=200,
    loss_function='YetiRank',
    verbose=50,
)
model.fit(train_pool)

preds = model.predict(test_pool)
print('YetiRank, nDCG:', compute_group_ndcg(y_test, preds, qid_test))

0:	total: 157ms	remaining: 31.2s
50:	total: 693ms	remaining: 2.02s
100:	total: 1.24s	remaining: 1.22s
150:	total: 1.76s	remaining: 570ms
199:	total: 2.26s	remaining: 0us
YetiRank, nDCG: 0.7479583118101537


In [5]:
model_pair = CatBoostRanker(
    loss_function='PairLogit',
    iterations=200,
    verbose=50,
)
model_pair.fit(train_pool)

pred_pair = model_pair.predict(test_pool)
print('PairLogit nDCG:', compute_group_ndcg(y_test, pred_pair, qid_test))

0:	learn: 0.6913301	total: 103ms	remaining: 20.4s
50:	learn: 0.6312476	total: 6.44s	remaining: 18.8s
100:	learn: 0.6119626	total: 13s	remaining: 12.8s
150:	learn: 0.5965459	total: 20.4s	remaining: 6.61s
199:	learn: 0.5833203	total: 27s	remaining: 0us
PairLogit nDCG: 0.7334023189337703


## Internet Math 2009

### Парсинг IMAT

In [6]:
def parse_imat(file_path):
    X, y, qid = [], [], []

    current_qid = 0
    max_feature = 0

    raw_features = []

    with open(file_path) as f:
        for line in f:
            parts = line.strip().split()

            label = float(parts[0])
            feats = {}

            for item in parts[1:]:
                if item.startswith('#'):
                    break

                idx, val = item.split(':')
                idx = int(idx) - 1
                val = float(val)

                feats[idx] = val
                max_feature = max(max_feature, idx)

            raw_features.append((feats, label))

    dim = max_feature + 1

    for feats, label in raw_features:
        vec = np.zeros(dim)

        for idx, val in feats.items():
            vec[idx] = val

        X.append(vec)
        y.append(label)
        qid.append(current_qid)

        if label == 2:
            current_qid += 1

    return np.array(X), np.array(y), np.array(qid)


X_train, y_train, qid_train = parse_imat('../data/imat/imat2009_train_new.txt')
X_test, y_test, qid_test = parse_imat('../data/imat/imat2009_test_new.txt')

train_pool = Pool(X_train, y_train, group_id=qid_train)
test_pool = Pool(X_test, y_test, group_id=qid_test)

In [7]:
def describe_dataset(X, y, qid):
    print('Примеров:', len(X))
    print('Фичей:', X.shape[1])
    print('Запросов:', len(np.unique(qid)))

    # print('\nРаспределение классов:')
    # unique, counts = np.unique(y, return_counts=True)
    # for u, c in zip(unique, counts):
    #     print(f'  label={u}: {c}')

    q_lengths = [np.sum(qid == q) for q in np.unique(qid)]
    print('\nДокументов на запрос:', np.mean(q_lengths))


describe_dataset(X_train, y_train, qid_train)

Примеров: 77714
Фичей: 245
Запросов: 24425

Документов на запрос: 3.181740020470829


### Обучение в 2 метода

In [8]:
random_preds = np.random.rand(len(y_test))
print('Random baseline, nDCG:', compute_group_ndcg(y_test, random_preds, qid_test))

Random baseline, nDCG: 0.801651742227663


In [9]:
model = CatBoostRanker(
    loss_function='YetiRank',
    iterations=200,
    verbose=50,
)
model.fit(train_pool)

pred = model.predict(test_pool)
print('YetiRank, nDCG:', compute_group_ndcg(y_test, pred, qid_test))


0:	total: 73ms	remaining: 14.5s
50:	total: 2.6s	remaining: 7.59s
100:	total: 5.14s	remaining: 5.04s
150:	total: 7.79s	remaining: 2.53s
199:	total: 10.5s	remaining: 0us
YetiRank, nDCG: 0.8984569519877776


In [10]:
model = CatBoostRanker(
    loss_function='PairLogit',
    iterations=200,
    verbose=50,
)
model.fit(train_pool)

pred = model.predict(test_pool)
print('PairLogit, nDCG:', compute_group_ndcg(y_test, pred, qid_test))

0:	learn: 0.6877132	total: 88.2ms	remaining: 17.5s
50:	learn: 0.5943432	total: 4.54s	remaining: 13.3s
100:	learn: 0.5679934	total: 9.17s	remaining: 8.98s
150:	learn: 0.5542650	total: 13.4s	remaining: 4.34s
199:	learn: 0.5444919	total: 17.5s	remaining: 0us
PairLogit, nDCG: 0.8886256666101443


## WikIR

### Данные er1k

In [11]:
def load_split(path):
    queries = pd.read_csv(os.path.join(path, 'queries.csv'))
    qrels = pd.read_csv(os.path.join(path, 'BM25.qrels.csv'))
    return queries, qrels


docs = pd.read_csv('../data/en1k/documents.csv')

train_queries, train_qrels = load_split('../data/en1k/training')
test_queries, test_qrels = load_split('../data/en1k/test')

doc_text = dict(zip(docs['id_right'], docs['text_right']))

tokenized_corpus = [text.split() for text in docs['text_right']]
bm25 = BM25Okapi(tokenized_corpus)

doc_ids = docs['id_right'].tolist()
doc_id_to_idx = {doc_id: i for i, doc_id in enumerate(doc_ids)}

set_terms = [set(doc) for doc in tokenized_corpus]
term_freq = [Counter(doc) for doc in tokenized_corpus]

term_doc_freq = Counter()
for doc in tokenized_corpus:
    unique_terms = set(doc)
    for term in unique_terms:
        term_doc_freq[term] += 1

idf_dict = {
    term: np.log((len(tokenized_corpus) + 1) / (1 + freq))
    for term, freq in term_doc_freq.items()
}

### BM25 и обучение CatBoost

In [12]:
def bm25_rank(query):
    scores = bm25.get_scores(query.split())
    order = np.argsort(scores)[::-1]
    return [doc_ids[i] for i in order[:100]]


def evaluate_bm25(queries, qrels):
    ndcgs = []

    for _, q_row in queries.iterrows():
        qid_val = q_row['id_left']
        query = q_row['text_left']

        scores = bm25.get_scores(query.split())
        order = np.argsort(scores)[::-1][:100]

        ranked_scores = scores[order]
        ranked_docs = [doc_ids[i] for i in order]

        rel_dict = (
            qrels[qrels.id_left == qid_val].set_index('id_right')['label'].to_dict()
        )

        true = [rel_dict.get(doc_id, 0) for doc_id in ranked_docs]

        if sum(true) > 0:
            ndcgs.append(ndcg_score([true], [ranked_scores]))

    return np.mean(ndcgs)


print('BM25 baseline, nDCG:', evaluate_bm25(test_queries, test_qrels))

BM25 baseline, nDCG: 0.8491612650241314


In [13]:
def extract_features(query, doc_id):
    q_terms = query.split()
    d_terms = tokenized_corpus[doc_id_to_idx[doc_id]]

    tf = sum(term_freq[doc_id_to_idx[doc_id]].get(t, 0) for t in q_terms)
    doc_len = len(d_terms)
    query_len = len(q_terms)

    overlap = len(set(q_terms) & set_terms[doc_id_to_idx[doc_id]])

    tf_norm = tf / (doc_len + 1)

    idf = sum(idf_dict.get(t, 0) for t in q_terms)

    positions = [i for i, t in enumerate(d_terms) if t in q_terms]
    proximity = max(positions) - min(positions) if len(positions) > 1 else doc_len

    return [tf, tf_norm, doc_len, query_len, overlap, idf, proximity]


def build_dataset(queries, qrels):
    X, y, qid = [], [], []

    for qid_val in queries['id_left']:
        query = queries.loc[queries.id_left == qid_val, 'text_left'].values[0]

        rel_docs = qrels[qrels.id_left == qid_val]
        rel_ids = set(rel_docs['id_right'])

        for _, row in rel_docs.iterrows():
            X.append(extract_features(query, row['id_right']))
            y.append(1)
            qid.append(qid_val)

        non_rel_ids = list(set(doc_ids) - rel_ids)
        sampled = np.random.choice(non_rel_ids, size=len(rel_docs), replace=False)

        for doc_id in sampled:
            X.append(extract_features(query, doc_id))
            y.append(0)
            qid.append(qid_val)

    return np.array(X), np.array(y), np.array(qid)


X_train, y_train, qid_train = build_dataset(train_queries, train_qrels)
X_test, y_test, qid_test = build_dataset(test_queries, test_qrels)

In [14]:
train_pool = Pool(X_train, y_train, group_id=qid_train)
test_pool = Pool(X_test, y_test, group_id=qid_test)

model = CatBoostRanker(
    loss_function='YetiRank',
    iterations=200,
    verbose=50,
)
model.fit(train_pool)

pred = model.predict(test_pool)
print('PairLogit, nDCG:', compute_group_ndcg(y_test, pred, qid_test))

0:	total: 262ms	remaining: 52.1s
50:	total: 7.1s	remaining: 20.7s
100:	total: 13.5s	remaining: 13.2s
150:	total: 19.9s	remaining: 6.46s
199:	total: 26s	remaining: 0us
PairLogit, nDCG: 0.9688840504920202


### Переранжирование top100

In [15]:
def rerank(query):
    scores = bm25.get_scores(query.split())
    top_idx = np.argsort(scores)[-100:]

    feats = []
    doc_ids_subset = []

    for idx in top_idx:
        doc_id = doc_ids[idx]

        feats.append(extract_features(query, doc_id))
        doc_ids_subset.append(doc_id)

    preds = model.predict(np.array(feats))

    order = np.argsort(preds)[::-1]
    return [doc_ids_subset[i] for i in order]


def evaluate(queries, qrels):
    ndcgs = []

    for _, q_row in queries.iterrows():
        qid_val = q_row['id_left']
        query = q_row['text_left']

        ranked = rerank(query)

        rel_dict = (
            qrels[qrels.id_left == qid_val].set_index('id_right')['label'].to_dict()
        )

        true = [rel_dict.get(doc_id, 0) for doc_id in ranked]
        pred = list(range(len(ranked), 0, -1))

        if sum(true) > 0:
            ndcgs.append(ndcg_score([true], [pred]))

    return np.mean(ndcgs)


print('Test, nDCG:', evaluate(test_queries, test_qrels))

Test, nDCG: 0.8472733286181132


### Реконструкция BM25

In [17]:
def extract_bm25_features(query, doc_id):
    q_terms = query.split()
    doc_idx = doc_id_to_idx[doc_id]
    d_terms = tokenized_corpus[doc_idx]

    tf = sum(term_freq[doc_idx].get(t, 0) for t in q_terms)
    doc_len = len(d_terms)

    idf = sum(idf_dict.get(t, 0) for t in q_terms)

    return [tf, idf, doc_len]


def build_dataset_bm25(queries, qrels):
    X, y, qid = [], [], []

    for qid_val in queries['id_left']:
        query = queries.loc[queries.id_left == qid_val, 'text_left'].values[0]

        rel_docs = qrels[qrels.id_left == qid_val]
        rel_ids = set(rel_docs['id_right'])

        for _, row in rel_docs.iterrows():
            X.append(extract_bm25_features(query, row['id_right']))
            y.append(1)
            qid.append(qid_val)

        non_rel_ids = list(set(doc_ids) - rel_ids)
        sampled = np.random.choice(non_rel_ids, size=len(rel_docs), replace=False)

        for doc_id in sampled:
            X.append(extract_bm25_features(query, doc_id))
            y.append(0)
            qid.append(qid_val)

    return np.array(X), np.array(y), np.array(qid)


X_train, y_train, qid_train = build_dataset_bm25(train_queries, train_qrels)
X_test, y_test, qid_test = build_dataset_bm25(test_queries, test_qrels)


model = CatBoostRanker(
    loss_function='YetiRank',
    iterations=200,
    verbose=50,
)
model.fit(Pool(X_train, y_train, group_id=qid_train))

pred = model.predict(X_test)
print('PairLogit, nDCG:', compute_group_ndcg(y_test, pred, qid_test))

0:	total: 202ms	remaining: 40.2s
50:	total: 7.06s	remaining: 20.6s
100:	total: 13.9s	remaining: 13.7s
150:	total: 20.7s	remaining: 6.7s
199:	total: 27s	remaining: 0us
PairLogit, nDCG: 0.9641452848179746


## MIRAGE

### Подгрузка данных

In [18]:
with open('../data/mirage/dataset.json', encoding='utf-8') as f:
    dataset = json.load(f)

with open('../data/mirage/doc_pool.json', encoding='utf-8') as f:
    doc_pool = json.load(f)

with open('../data/mirage/oracle.json', encoding='utf-8') as f:
    oracle = json.load(f)


queries = {item['query_id']: item['query'] for item in dataset}

doc_pool_by_qid = defaultdict(list)
for doc in doc_pool:
    doc_pool_by_qid[doc['mapped_id']].append(doc)

oracle_docs = {k: v for k, v in oracle.items()}

In [19]:
def extract_features(query, doc):  # noqa
    text = doc['doc_chunk']
    title = doc.get('title', '')

    q_terms = query.lower().split()
    d_terms = text.lower().split()
    t_terms = title.lower().split()

    tf_body = sum(d_terms.count(t) for t in q_terms)
    tf_title = sum(t_terms.count(t) for t in q_terms)

    overlap_body = len(set(q_terms) & set(d_terms))
    overlap_title = len(set(q_terms) & set(t_terms))

    return [
        tf_body,
        tf_title,
        len(d_terms),
        len(t_terms),
        overlap_body,
        overlap_title,
    ]


X, y, qid = [], [], []

for q_idx, (qid_val, query) in enumerate(tqdm(queries.items())):
    if qid_val not in doc_pool_by_qid:
        continue

    docs = doc_pool_by_qid[qid_val]

    if qid_val in oracle_docs:
        X.append(extract_features(query, oracle_docs[qid_val]))
        y.append(1)
        qid.append(q_idx)

    for doc in docs:
        if doc['doc_chunk'] == oracle_docs.get(qid_val, {}).get('doc_chunk'):
            continue

        X.append(extract_features(query, doc))
        y.append(0)
        qid.append(q_idx)

X = np.array(X)
y = np.array(y)
qid = np.array(qid)

unique_qids = np.unique(qid)
train_q, test_q = train_test_split(unique_qids, test_size=0.2, random_state=42)

train_mask = np.isin(qid, train_q)
test_mask = np.isin(qid, test_q)

X_train, y_train, qid_train = X[train_mask], y[train_mask], qid[train_mask]
X_test, y_test, qid_test = X[test_mask], y[test_mask], qid[test_mask]

100%|██████████| 7560/7560 [00:01<00:00, 5835.10it/s]


### Обучение и оценка

In [20]:
train_pool = Pool(X_train, y_train, group_id=qid_train)
test_pool = Pool(X_test, y_test, group_id=qid_test)

model = CatBoostRanker(
    iterations=200,
    loss_function='YetiRank',
    verbose=50,
)

model.fit(train_pool)

0:	total: 16.4ms	remaining: 3.26s
50:	total: 461ms	remaining: 1.35s
100:	total: 914ms	remaining: 896ms
150:	total: 1.38s	remaining: 446ms
199:	total: 1.81s	remaining: 0us


CatBoostRanker(iterations=200, loss_function='YetiRank', verbose=50)

In [21]:
preds = model.predict(test_pool)
print('MIRAGE, nDCG:', compute_group_ndcg(y_test, preds, qid_test))

MIRAGE, nDCG: 0.761392249657054
